# momentum-buffer-update — faded example 3: Initialize the momentum buffer on the first step when it doesn't exist yet

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `momentum-buffer-update`. Running the beacon reports progress on the `Optimizer: Momentum buffer` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Momentum buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`momentum-buffer-update`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "momentum-buffer-update"
DD_SUBTOPIC = "Optimizer: Momentum buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Real optimizers lazily initialize the momentum buffer: the first time a parameter receives a gradient, the buffer is created as a clone of that gradient (rather than zeros). This is equivalent to setting `b₀ = g₀` on step one and then applying `b ← μ·b + g` for all subsequent steps. A None-check on the buffer is the standard implementation pattern.

## Faded exercise 3

Implement `momentum_lazy_step(state, grad, mu)` where `state` is a dict that may or may not have a `'buf'` key.

If `state['buf']` is absent (or `None`): **initialize** `state['buf'] = grad.clone()` (no momentum on first step).
Otherwise: update **in-place**: `state['buf'].copy_(mu * state['buf'] + grad)`.

Return `state['buf']` (the effective gradient).

Your task: **fill in the initialization branch and the in-place update branch**.

**Fill in:** The None-check branch that either clones grad into state['buf'] (first step) or applies b.copy_(mu*b + g) in-place (subsequent steps), then returns state['buf'].

In [ ]:
import torch

def momentum_lazy_step(state: dict, grad: torch.Tensor, mu: float) -> torch.Tensor:
    raise NotImplementedError()  # TODO: The None-check branch that either clones grad into state['buf'] (first step) or applies b.copy_(mu*b + g) in-place (subsequent steps), then returns state['buf'].

def _test():
    import torch
    g1 = torch.tensor([1.0, 2.0])
    g2 = torch.tensor([0.5, -1.0])
    state = {}
    # First step: buf initialised to g1 (no momentum yet)
    out1 = momentum_lazy_step(state, g1, mu=0.9)
    assert torch.allclose(out1, g1), f"first step should be g, got {out1}"
    assert 'buf' in state
    assert out1 is state['buf']
    # Second step: buf = 0.9*g1 + g2
    out2 = momentum_lazy_step(state, g2, mu=0.9)
    expected = 0.9 * g1 + g2
    assert torch.allclose(out2, expected, atol=1e-6), f"got {out2}, expected {expected}"


def _test():
    import torch
    g1 = torch.tensor([1.0, 2.0])
    g2 = torch.tensor([0.5, -1.0])
    g3 = torch.tensor([-0.2, 0.3])
    state = {}
    out1 = momentum_lazy_step(state, g1, mu=0.9)
    assert torch.allclose(out1, g1), f"first step: got {out1}"
    assert out1 is state['buf'], "buf should be in state"
    out2 = momentum_lazy_step(state, g2, mu=0.9)
    exp2 = 0.9 * g1 + g2
    assert torch.allclose(out2, exp2, atol=1e-6), f"step2: got {out2}"
    out3 = momentum_lazy_step(state, g3, mu=0.9)
    exp3 = 0.9 * exp2 + g3
    assert torch.allclose(out3, exp3, atol=1e-6), f"step3: got {out3}"
    # buffer in state should be same object as return value
    assert out3 is state['buf']


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch

def momentum_lazy_step(state: dict, grad: torch.Tensor, mu: float) -> torch.Tensor:
    if state.get('buf') is None:
        state['buf'] = grad.clone()
    else:
        state['buf'].copy_(mu * state['buf'] + grad)
    return state['buf']

def _test():
    import torch
    g1 = torch.tensor([1.0, 2.0])
    g2 = torch.tensor([0.5, -1.0])
    state = {}
    out1 = momentum_lazy_step(state, g1, mu=0.9)
    assert torch.allclose(out1, g1)
    assert 'buf' in state
    assert out1 is state['buf']
    out2 = momentum_lazy_step(state, g2, mu=0.9)
    expected = 0.9 * g1 + g2
    assert torch.allclose(out2, expected, atol=1e-6)
```
</details>